In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd


# ============================================================
# 1. PROJECT INITIALIZATION
# ============================================================

def prepare_directories():
    BASE_DIR = os.getcwd()
    dirs = {
        "BASE": BASE_DIR,
        "DATA": os.path.join(BASE_DIR, "data"),
        "OUT": os.path.join(BASE_DIR, "output"),
        "WEIGHT": os.path.join(BASE_DIR, "weights"),
        "SRC": os.path.join(BASE_DIR, "src"),
        "SHP": os.path.join(BASE_DIR, "shp")
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    print("Project structure ready")
    print("Base directory:", BASE_DIR)
    return dirs


# ============================================================
# 2. COUNT ROWS IN A CSV
# ============================================================

def count_rows(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            n = sum(1 for _ in f)
        print(f"Total rows: {n}")
        return n
    except Exception as e:
        print(f"Error: {e}")
        return None


# ============================================================
# 3. CLEAN BUILDING CSV: REMOVE DUPLICATES BY AREA
# ============================================================

def clean_building_identity(dirs):
    input_path = os.path.join(dirs["DATA"], "0106_building_indentity_neighborhoodsname.csv")
    output_path = os.path.join(dirs["OUT"], "0106_building_indentity_neighborhoodsname_cleaned.csv")

    df = pd.read_csv(input_path)
    count_rows(input_path)

    id_col = "identificatie"
    area_col = "Shape_Area"

    if df.duplicated(subset=[id_col], keep=False).any():
        print(f"Column '{id_col}' has duplicate values")
        df_cleaned = df.loc[df.groupby(id_col)[area_col].idxmax()]
        df_cleaned.to_csv(output_path, index=False, encoding="utf-8-sig")
        print(f"Duplicates cleaned by '{area_col}'.")
        print("Saved file:", output_path)
        print("Total rows after cleaning:", len(df_cleaned))
    else:
        print(f"Column '{id_col}' has no duplicate values")
        df_cleaned = df.copy()
        df_cleaned.to_csv(output_path, index=False, encoding="utf-8-sig")

    return output_path


# ============================================================
# 4. DROP ROWS WHERE buurtcode IS NaN
# ============================================================

def drop_nan_buurtcode(dirs, cleaned_building_path):
    df = pd.read_csv(cleaned_building_path)

    cleaned_file = os.path.join(dirs["OUT"], "0106_building_indentity_neighborhoodsname_cleaned_dropnancode.csv")
    removed_file = os.path.join(dirs["OUT"], "removed_rows_data_nobuurtcode.csv")

    removed = df[df["buurtcode"].isna()]
    df_cleaned = df.dropna(subset=["buurtcode"])

    df_cleaned.to_csv(cleaned_file, index=False, encoding="utf-8-sig")
    removed.to_csv(removed_file, index=False, encoding="utf-8-sig")

    print("Cleaned file:", cleaned_file)
    print("Removed rows file:", removed_file)
    print("Rows removed:", len(removed))
    print("Columns:", df_cleaned.columns.tolist())
    print("Total rows after cleaning:", len(df_cleaned))
    print("Unique buurtcode:", df_cleaned["buurtcode"].nunique())

    return cleaned_file


# ============================================================
# 5. FILTER SHAPEFILE TO MATCH buurtcode IN CSV
# ============================================================

def filter_shapefile_by_csv(dirs, cleaned_csv):
    shp_path = os.path.join(dirs["DATA"], "main_buurten selection.shp")
    gdf = gpd.read_file(shp_path)

    df = pd.read_csv(cleaned_csv)
    codes = df["buurtcode"].tolist()

    filtered = gdf[gdf["buurtcode"].isin(codes)]

    row_count = len(filtered)
    output_name = f"{row_count}_filtered_neighborhood_boundaries.shp"
    output_path = os.path.join(dirs["SHP"], output_name)

    filtered.to_file(output_path, driver="ESRI Shapefile")

    print("Filtered shapefile saved:", output_path)
    print("Total selected neighborhoods:", row_count)

    return output_path


# ============================================================
# 6. SLIM ENERGY LABEL DATA
# ============================================================

def slim_energylabel(dirs):
    input_path = os.path.join(dirs["DATA"], "1205_energy_label_full.csv")
    output_path = os.path.join(dirs["OUT"], "1205_energylabel_2022_slim.csv")

    print("Slimming energy label data")

    df = pd.read_csv(input_path, dtype=str)

    # Select required columns
    cols = ["identifica", "aantal_ver", "dominant_label", "Shape_Area"]
    df_slim = df[cols].copy()

    # Convert Shape_Area to numeric
    df_slim["Shape_Area"] = pd.to_numeric(df_slim["Shape_Area"], errors="coerce")

    # Remove rows with missing Shape_Area
    df_slim = df_slim.dropna(subset=["Shape_Area"])

    # Keep the largest record for each identificatie
    df_slim = (
        df_slim.sort_values(["identifica", "Shape_Area"], ascending=[True, False])
        .drop_duplicates(subset="identifica", keep="first")
    )

    # Rename columns
    df_slim = df_slim.rename(columns={
        "identifica": "identificatie",
        "aantal_ver": "aant_verblijfsobj"
    })

    # Drop temporary column
    df_slim = df_slim.drop(columns=["Shape_Area"])

    df_slim.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("Slim energy label saved:", output_path)
    print("Unique buildings:", len(df_slim))

    # Check if duplicate IDs still exist
    dup = df_slim["identificatie"].duplicated().sum()
    print("Duplicate identificatie after slimming:", dup)

    return output_path


# ============================================================
# 7. MERGE BUILDING CSV WITH ENERGY LABEL
# ============================================================

def merge_building_energy(dirs, building_file, energy_slim):
    df1 = pd.read_csv(building_file, dtype=str)
    df2 = pd.read_csv(energy_slim, dtype=str)

    df1["identificatie"] = df1["identificatie"].str.zfill(16)
    df2["identificatie"] = df2["identificatie"].str.zfill(16)

    merged = pd.merge(df1, df2, on="identificatie", how="left")

    output = os.path.join(dirs["OUT"], "energylabel_building_identity_neighborhoods.csv")
    merged.to_csv(output, index=False, encoding="utf-8-sig")

    print("Merged dataset saved:", output)
    print("Total rows:", len(merged))

    return output


# ============================================================
# 8. BUILDING CHARACTERISTICS SUMMARY
# ============================================================

def summarize_building_characteristics(dirs, merged_building):
    df = pd.read_csv(merged_building, dtype=str)

    df.columns = df.columns.str.strip()

    numeric_cols = ["Shape_Area", "gebouwhoogte", "aant_verblijfsobj"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["total_area_perbuilding"] = df["Shape_Area"] * (df["gebouwhoogte"] / 3.0)

    building_id_candidates = ["gebouw_id", "bag_pand_id", "building_id", "pand_id"]
    building_col = next((c for c in building_id_candidates if c in df.columns), None)
    print("Using building ID:", building_col)

    if building_col:
        grouped = df.groupby("buurtcode").agg(
            building_count_total=(building_col, "nunique"),
            ground_area=("Shape_Area", "sum"),
            total_area=("total_area_perbuilding", "sum"),
            total_verblijfsobj=("aant_verblijfsobj", "sum")
        )
    else:
        grouped = df.groupby("buurtcode").agg(
            building_count_total=("buurtcode", "size"),
            ground_area=("Shape_Area", "sum"),
            total_area=("total_area_perbuilding", "sum"),
            total_verblijfsobj=("aant_verblijfsobj", "sum")
        )

    g = df.groupby(["buurtcode", "dominant_label"], dropna=False)

    if building_col:
        label_stats = g.agg(
            bld_count=(building_col, "nunique"),
            hh_count=("aant_verblijfsobj", "sum")
        )
    else:
        label_stats = g.agg(
            bld_count=("dominant_label", "size"),
            hh_count=("aant_verblijfsobj", "sum")
        )

    A_labels = ["A", "A+", "A++", "A+++", "A++++", "A+++++"]

    bld_A = (
        label_stats.loc[label_stats.index.get_level_values("dominant_label").isin(A_labels)]
        .groupby("buurtcode")["bld_count"]
        .sum()
    )

    hh_A = (
        label_stats.loc[label_stats.index.get_level_values("dominant_label").isin(A_labels)]
        .groupby("buurtcode")["hh_count"]
        .sum()
    )

    result = grouped.copy()
    result["bld_A_total"] = bld_A.reindex(result.index).fillna(0).astype(float)
    result["hh_A_total"] = hh_A.reindex(result.index).fillna(0).astype(float)

    result["bld_A_share"] = result["bld_A_total"] / result["building_count_total"]
    result.loc[result["building_count_total"] == 0, "bld_A_share"] = np.nan

    result["hh_A_share"] = result["hh_A_total"] / result["total_verblijfsobj"]
    result.loc[result["total_verblijfsobj"] == 0, "hh_A_share"] = np.nan

    result["bld_A_pct"] = (result["bld_A_share"] * 100).round(2)
    result["hh_A_pct"] = (result["hh_A_share"] * 100).round(2)

    final_cols = [
        "building_count_total",
        "ground_area",
        "total_area",
        "total_verblijfsobj",
        "bld_A_total",
        "bld_A_pct",
        "hh_A_total",
        "hh_A_pct"
    ]

    final = result[final_cols].reset_index()

    output = os.path.join(dirs["OUT"], "neighborhoods_summary_building_characteristics.csv")
    final.to_csv(output, index=False, encoding="utf-8-sig")

    print("Building summary saved:", output)

    return output


# ============================================================
# 9. CLEAN BUURT TABLE FROM EXCEL
# ============================================================

def process_buurt_excel(dirs):
    input_file = os.path.join(dirs["DATA"], "kwb-2022.xlsx")
    output_file = os.path.join(dirs["OUT"], "buurt_filtered_table_columns_cleaned.csv")

    df = pd.read_excel(input_file, dtype=str)

    df = df[df["recs"] == "Buurt"].copy()
    print("Rows after filtering Buurt:", len(df))

    # Select relevant socio-economic variables
    selected_columns = [
        "regio", "gm_naam", "gwb_code", "a_inw", "a_hh",
        "g_hhgro", "bev_dich", "a_woning", "p_1gezw", "p_mgezw",
        "p_bewndw", "p_leegsw", "p_bjj2k", "p_bjo2k",
        "g_ele", "g_gas", "p_stadsv", "p_ink_hi", "a_opp_ha",
        "a_lan_ha", "ste_oad", "ste_mvs"
    ]

    existing = [c for c in selected_columns if c in df.columns]
    df = df[existing].copy()

    if "g_hhgro" in df.columns:
        df["g_hhgro"] = (
            df["g_hhgro"]
            .replace(".", np.nan)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )

    nan_like = ["nan", " ", ".", "", "NaN", "NONE"]
    df = df.replace(nan_like, np.nan)

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Buurt table cleaned and saved:", output_file)

    return output_file


# ============================================================
# 10. DROP NA IN SOCIO-ECONOMIC VARIABLES
# ============================================================

def clean_buurt_table(dirs, input_file):
    output_file = os.path.join(dirs["OUT"], "buurt_filtered_table_cleaned_selected.csv")

    df = pd.read_csv(input_file, dtype=str)
    df = df.replace(["nan", " ", ".", ""], np.nan)

    for col in ["g_ele", "g_gas", "p_stadsv"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    cols_to_check = ["p_1gezw", "p_bewndw", "p_bjj2k", "p_ink_hi"]
    df = df.dropna(subset=[c for c in cols_to_check if c in df.columns])

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Cleaned socio-economic table saved:", output_file)

    return output_file


# ============================================================
# 11. COMPUTE ENERGY CONSUMPTION
# ============================================================

def compute_energy_consumption(dirs, input_file):
    output_file = os.path.join(dirs["OUT"], "neighborhoods_energy_table.csv")

    df = pd.read_csv(input_file, dtype=str)

    numeric_cols = [
        "g_ele", "g_gas", "p_stadsv", "p_ink_hi",
        "a_woning", "p_bewndw", "a_inw"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "g_hhgro" in df.columns:
        df["g_hhgro"] = df["g_hhgro"].astype(str).str.replace(",", ".", regex=False)
        df["g_hhgro"] = pd.to_numeric(df["g_hhgro"], errors="coerce")

    df["g_ele"] = df["g_ele"].fillna(0)
    df["g_gas"] = df["g_gas"].fillna(0)
    df["p_stadsv"] = df["p_stadsv"].fillna(0)

    df["ele_total"] = df["g_ele"] * df["a_woning"] * (df["p_bewndw"] / 100)
    df["gas_total"] = df["g_gas"] * df["a_woning"] * (df["p_bewndw"] / 100)
    df["ec_district_heating"] = 12483 * df["a_woning"] * (df["p_stadsv"] / 100)

    df["ec_total"] = (
        df["ele_total"]
        + df["gas_total"] * (35.2 / 3.6)
        + df["ec_district_heating"]
    )

    df["ec_inten_percapita"] = np.where(
        df["a_inw"] > 0,
        df["ec_total"] / df["a_inw"],
        np.nan
    )

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Energy table saved:", output_file)

    return output_file


# ============================================================
# 12. MERGE ENERGY AND BUILDING CHARACTERISTICS
# ============================================================

def merge_energy_building(dirs, building_summary, energy_table):
    df_A = pd.read_csv(building_summary, dtype=str)
    df_E = pd.read_csv(energy_table, dtype=str)

    output = os.path.join(dirs["OUT"], "neighborhoods_energy_building_characteristics_merged.csv")

    merged = pd.merge(
        df_A,
        df_E,
        left_on="buurtcode",
        right_on="gwb_code",
        how="inner"
    )

    merged.to_csv(output, index=False, encoding="utf-8-sig")

    print("Merged final table saved:", output)
    print("Total rows:", len(merged))

    return output


# ============================================================
# 13. COMPUTE MORPHOLOGY METRICS
# ============================================================

def compute_morphology(dirs, merged_file):
    df = pd.read_csv(merged_file)

    land_area_m2 = df["a_lan_ha"] * 10000

    df["building_Density"] = df["ground_area"] / land_area_m2 * 100
    df["building_FAR"] = df["total_area"] / land_area_m2
    df["OSR"] = (100 - df["building_Density"]) / df["building_FAR"]
    df["woning_density"] = df["a_woning"] / df["a_lan_ha"]
    df["ave_floor"] = df["building_FAR"] / (df["building_Density"] / 100)
    df["building_inten"] = df["a_woning"] / df["building_count_total"]

    output = os.path.join(dirs["OUT"], "neighborhoods_energy_building_characteristics_merged_more.csv")
    df.to_csv(output, index=False, encoding="utf-8-sig")

    print("Morphology metrics saved:", output)

    return output


# ============================================================
# 14. EXPORT FILTERED SHAPEFILE BASED ON FINAL TABLE
# ============================================================

def filter_final_shapefile(dirs, final_csv):
    shp_path = os.path.join(dirs["DATA"], "main_buurten selection.shp")
    gdf = gpd.read_file(shp_path)

    df = pd.read_csv(final_csv)
    codes = df["buurtcode"].tolist()

    filtered = gdf[gdf["buurtcode"].isin(codes)]

    row_count = len(filtered)
    output_shp = os.path.join(
        dirs["SHP"],
        f"{row_count}_filtered_neighborhood_boundaries_final.shp"
    )

    filtered.to_file(output_shp, encoding="utf-8")

    print("Final shapefile saved:", output_shp)
    print("Total rows:", row_count)

    return output_shp


# ============================================================
# 15. MAIN WORKFLOW
# ============================================================

def main():
    dirs = prepare_directories()

    # Step 1: Clean building dataset
    building_cleaned = clean_building_identity(dirs)
    building_no_nan = drop_nan_buurtcode(dirs, building_cleaned)
    filter_shapefile_by_csv(dirs, building_no_nan)

    # Step 2: Process energy label data
    energy_slim = slim_energylabel(dirs)
    building_energy_merged = merge_building_energy(dirs, building_no_nan, energy_slim)

    # Step 3: Summarize building characteristics
    building_summary = summarize_building_characteristics(dirs, building_energy_merged)

    # Step 4: Process buurt socio-economic table
    buurt_initial = process_buurt_excel(dirs)
    buurt_cleaned = clean_buurt_table(dirs, buurt_initial)
    energy_table = compute_energy_consumption(dirs, buurt_cleaned)

    # Step 5: Merge building and energy data
    merged_BE = merge_energy_building(dirs, building_summary, energy_table)

    # Step 6: Compute morphology metrics
    final_csv = compute_morphology(dirs, merged_BE)

    # Step 7: Export final filtered shapefile
    filter_final_shapefile(dirs, final_csv)

    print("\n========================")
    print(" WORKFLOW COMPLETED ")
    print("========================")


if __name__ == "__main__":
    main()